# Task 3 — A/B Hypothesis Testing

This analysis statistically evaluates whether significant differences exist across customer and geographic risk segments within the insurance portfolio.

The objective is to validate or reject key business hypotheses related to:
- insurance risk,
- claim severity,
- claim frequency,
- and portfolio profitability.

The findings from this analysis support evidence-based pricing strategies and risk segmentation decisions.

In [1]:
import sys
import os
sys.path.append(os.path.abspath(".."))

import pandas as pd
import numpy as np

from scipy.stats import ttest_ind
from scipy.stats import chi2_contingency

from src.data_loader import load_data
from src.hypothesis_tests import (
    run_ttest,
    run_chi_square
)

In [2]:
df = load_data("../data/insurance_data.csv")

df.head()

,CustomerID,Age,Gender,Province,VehicleType,AnnualIncome,RiskScore,AnnualPremium,Deductible,NCD,...,Claimed,ClaimAmount,TotalPremium,TotalClaims,CoverType,AutoMake,VehicleModel,CustomValueEstimate,ZipCode,TransactionDate
0,AC-100000,56,Male,Addis Ababa,Sedan,147270,61,2346,500,30,...,False,0.0,2346,0.0,Comprehensive,Lifan,620,32238,10002,2024-05-10
1,AC-100001,69,Female,Addis Ababa,SUV,74640,57,2334,500,0,...,True,9883.0,2334,9883.0,Comprehensive,Suzuki,Grand Vitara,52510,10001,2024-08-13
2,AC-100002,46,Male,Oromia,Sedan,70555,42,1697,250,20,...,False,0.0,1697,0.0,Third Party Fire & Theft,Lifan,620,26523,20001,2025-03-17
3,AC-100003,32,Female,Somali,Sedan,89398,63,2370,500,20,...,True,12134.0,2370,12134.0,Comprehensive,Toyota,Corolla,27036,40005,2025-03-17
4,AC-100004,60,Female,Tigray,SUV,78475,69,2582,500,0,...,False,0.0,2582,0.0,Comprehensive,Toyota,RAV4,58348,50002,2024-11-10


## Margin Calculation

Margin represents portfolio profitability and is calculated as:

Margin = TotalPremium − TotalClaims

Positive margins indicate profitable policies, while negative margins indicate potential underwriting losses.

In [3]:
df["Margin"] = (
    df["TotalPremium"] - df["TotalClaims"]
)

## Claim Severity Dataset

Claim severity measures the average claim amount among policies that recorded at least one claim.

Only policies with non-zero claims are included in severity analysis.

In [4]:
claims_only = df[
    df["TotalClaims"] > 0
]

## Hypothesis 1 — Risk Differences Across Provinces

### Null Hypothesis (H₀)
There are no significant risk differences across provinces.

### Groups
- Group A (Control): Addis Abeba
- Group B (Test): Oromia

### KPI
Claim Severity

### Statistical Test
Independent Two-Sample t-test

The selected provinces contain large and relatively balanced sample sizes, improving the reliability of the statistical comparison.

In [5]:
province_df = claims_only[
    claims_only["Province"].isin(
        ["Addis Ababa", "Oromia"]
    )
]

province_a = province_df[
    province_df["Province"] == "Addis Ababa"
]["TotalClaims"]

province_b = province_df[
    province_df["Province"] == "Oromia"
]["TotalClaims"]

print("Addis Ababa:", len(province_a))
print("Oromia:", len(province_b))

Addis Ababa: 559
Oromia: 377


In [6]:
province_result = run_ttest(
    province_a,
    province_b
)

province_p = province_result["p_value"]

province_result

{'t_statistic': np.float64(-0.7955637073077246),
 'p_value': np.float64(0.42654644476735115)}

In [7]:
if province_p < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

Fail to Reject H0


## Hypothesis 2 — Risk Differences Between Zip Codes

### Null Hypothesis (H₀)
There are no significant risk differences between zip codes.

### Groups
- Group A (Control): Zip Code 10001
- Group B (Test): Zip Code 10004

### KPI
Claim Severity

### Statistical Test
Independent Two-Sample t-test

The selected zip codes belong to the same regional cluster and contain sufficiently balanced sample sizes.

In [8]:
zip_df = claims_only[
    claims_only["ZipCode"].isin(
        [10001, 10004]
    )
]

zip_a = zip_df[
    zip_df["ZipCode"] == 10001
]["TotalClaims"]

zip_b = zip_df[
    zip_df["ZipCode"] == 10004
]["TotalClaims"]

print("10001:", len(zip_a))
print("10004:", len(zip_b))

10001: 121
10004: 109


In [9]:
zip_result = run_ttest(
    zip_a,
    zip_b
)

zip_p = zip_result["p_value"]

zip_result

{'t_statistic': np.float64(0.3068096635098361),
 'p_value': np.float64(0.7592754462470345)}

In [10]:
if zip_p < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

Fail to Reject H0


## Hypothesis 3 — Margin Differences Between Zip Codes

### Null Hypothesis (H₀)
There is no significant profitability difference between zip codes.

### Groups
- Group A (Control): Zip Code 10001
- Group B (Test): Zip Code 10004

### KPI
Margin

### Statistical Test
Independent Two-Sample t-test

In [11]:
margin_a = df[
    df["ZipCode"] == 10001
]["Margin"]

margin_b = df[
    df["ZipCode"] == 10004
]["Margin"]

print("10001:", len(margin_a))
print("10004:", len(margin_b))

10001: 710
10004: 733


In [12]:
margin_result = run_ttest(
    margin_a,
    margin_b
)

margin_p = margin_result["p_value"]

margin_result

{'t_statistic': np.float64(-1.3303445489740426),
 'p_value': np.float64(0.1836165233277263)}

In [13]:
if margin_p < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

Fail to Reject H0


## Hypothesis 4 — Risk Differences Between Women and Men

### Null Hypothesis (H₀)
There is no significant risk difference between women and men.

### KPI
Claim Frequency

### Statistical Test
Chi-Square Test of Independence

Claim frequency is evaluated using claim occurrence across gender categories.

In [14]:
gender_df = df[
    df["Gender"].isin(
        ["Male", "Female"]
    )
]

contingency_table = pd.crosstab(
    gender_df["Gender"],
    gender_df["Claimed"]
)

contingency_table

Claimed,False,True
Gender,,
Female,4348,790
Male,4117,745


In [15]:
gender_result = run_chi_square(
    contingency_table
)

gender_p = gender_result["p_value"]

gender_result

{'chi2_statistic': np.float64(0.0020563626938139516),
 'p_value': np.float64(0.9638306173980254)}

In [16]:
if gender_p < 0.05:
    print("Reject H0")
else:
    print("Fail to Reject H0")

Fail to Reject H0


## Hypothesis Testing Summary

The following table summarizes:
- the hypothesis tested,
- KPI selected,
- statistical test used,
- p-value,
- and final decision.

In [17]:
results = pd.DataFrame({
    "Hypothesis": [
        "Province Risk Difference",
        "Zip Code Risk Difference",
        "Zip Code Margin Difference",
        "Gender Risk Difference"
    ],
    "KPI": [
        "Claim Severity",
        "Claim Severity",
        "Margin",
        "Claim Frequency"
    ],
    "Test Used": [
        "T-Test",
        "T-Test",
        "T-Test",
        "Chi-Square"
    ],
    "P-Value": [
        province_p,
        zip_p,
        margin_p,
        gender_p
    ],
    "Decision": [
        "Reject H0" if province_p < 0.05 else "Fail to Reject H0",
        "Reject H0" if zip_p < 0.05 else "Fail to Reject H0",
        "Reject H0" if margin_p < 0.05 else "Fail to Reject H0",
        "Reject H0" if gender_p < 0.05 else "Fail to Reject H0"
    ]
})

results

,Hypothesis,KPI,Test Used,P-Value,Decision
0,Province Risk Difference,Claim Severity,T-Test,0.426546,Fail to Reject H0
1,Zip Code Risk Difference,Claim Severity,T-Test,0.759275,Fail to Reject H0
2,Zip Code Margin Difference,Margin,T-Test,0.183617,Fail to Reject H0
3,Gender Risk Difference,Claim Frequency,Chi-Square,0.963831,Fail to Reject H0


# Business Recommendations

### Province Risk Differences
If significant differences are detected across provinces, ACIS should consider province-specific pricing adjustments and regional underwriting policies.

### Zip Code Risk Differences
Significant zip code variation may indicate localized risk exposure requiring geographic segmentation strategies.

### Margin Differences
Profitability differences between zip codes may suggest that some regions are underpriced or overpriced relative to actual risk.

### Gender Risk Differences
If claim frequency differs significantly by gender, demographic segmentation may improve portfolio pricing accuracy.

# Conclusion

The hypothesis testing analysis evaluated whether statistically significant differences exist across major insurance risk segments.

The results provide evidence for whether geography, zip code, profitability, and gender influence claim behavior and portfolio performance.

These findings establish a statistical foundation for future pricing optimization, underwriting improvements, and predictive risk modeling initiatives.